In [2]:
# 5_normalise_ukhls.ipynb
#
# Z-score normalises the feature-engineered UKHLS data ready for K-Means.
# Reads from 4_feature_eng_ukhls output — all clipping, recoding and
# column selection is already done upstream.
#
# Steps:
#   1. Load o_indresp_feature_eng.pkl
#   2. StandardScaler (Z-score) on all feature columns
#   3. Save as normalized.pkl

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_PKL  = "../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"
OUTPUT_PKL = "../data/5_normalise_ukhls/normalized.pkl"

# ── 1. Load ───────────────────────────────────────────────────────────────────
print(f"Loading {INPUT_PKL} ...")
if not os.path.exists(INPUT_PKL):
    raise FileNotFoundError(
        f"{INPUT_PKL} not found — run 4_feature_eng_ukhls.ipynb first."
    )

df = pd.read_pickle(INPUT_PKL)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

feature_cols = [c for c in df.columns if c != 'pidp']
print(f"Features to normalise: {len(feature_cols)}")

# ── 2. Z-score normalise ──────────────────────────────────────────────────────
print("\nApplying StandardScaler (Z-score) ...")
scaler = StandardScaler()

df_norm = pd.DataFrame(index=df.index)
df_norm['pidp'] = df['pidp']
df_norm[feature_cols] = scaler.fit_transform(df[feature_cols].astype(np.float32))

print("  Done.")
print(f"  Mean across all features: {df_norm[feature_cols].mean().mean():.4f}  (expected ~0)")
print(f"  Std  across all features: {df_norm[feature_cols].std().mean():.4f}   (expected ~1)")

# ── 3. Save ───────────────────────────────────────────────────────────────────
df_norm.to_pickle(OUTPUT_PKL, protocol=5)
print(f"\nDone. Normalised matrix saved to {OUTPUT_PKL}")
print(f"Shape: {df_norm.shape}")


Loading ../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl ...
Loaded 47,354 rows × 27 columns
Features to normalise: 26

Applying StandardScaler (Z-score) ...
  Done.
  Mean across all features: -0.0000  (expected ~0)
  Std  across all features: 1.0000   (expected ~1)

Done. Normalised matrix saved to ../data/5_normalise_ukhls/normalized.pkl
Shape: (47354, 27)
